# Cluster number commutator cost using up to 4-rdm

In [3]:
import numpy as np

# Molecule input parameters
molecule = 'h2o'              # supports: h2, h2o, n2, lih, h4_linear, h4_square
bond_length = 1.            # Angstrom
basis_set = 'sto3g'              # 'sto3g', '6-31g', ...
cluster_matrix = np.array([
        [1, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 0]
    ])

# DMRG parameters
bond_dim = 50 # 500
n_sweeps = 10 # 20

In [4]:
# Verify input validity

# Calculate the sum of each column (one column per orbital)
column_sums = np.sum(cluster_matrix, axis=0)
if not np.all(np.isin(column_sums, [0, 1])):
    invalid_columns = np.where((column_sums != 0) & (column_sums != 1))[0]
    raise ValueError(f"Error: Orbitals {invalid_columns} should appear in at most one cluster.")
if any([all(bit == 0 for bit in cluster) for cluster in cluster_matrix]):
    raise ValueError(f"Error: There is one or more empty clusters.")
# We do accept redundant clustering.
#    if np.all(column_sums == 1):
#        raise ValueError(f"Error: The clusters cover all orbitals. Remove one cluster to avoid redundancy.")
    num_clusters = len(cluster_matrix)

In [5]:
# ============================================================================
# SECTION 1: Get MOs and mps
# ============================================================================
import sys
from pathlib import Path
import hashlib
import json
sys.path.insert(0, '..') 
import src.cluster_number_operators as cluster_nums
import pyscf
from chemistry import get_geometry_and_description
from src.dmrg_solver import Block2DMRGSolver, DMRGConfig, solve_or_load_ground_state
from math import comb

# --- Step 1.1: Build molecule and run HF - we need an ONB and Hamiltonian ---
geometry, _ = get_geometry_and_description(molecule, bond_length)
mol = pyscf.M(atom=geometry, basis=basis_set)
mf = pyscf.scf.RHF(mol)
mf.kernel()

norb, nelec = mol.nao, mol.nelec
# assert norb == cluster_matrix.shape[1], f"Number of columns of cluster_matrix = {cluster_matrix.shape[1]} does not match number of orbitals = {norb}"
print(f"Number of orbitals: {norb}")
print(f"Number of elecs: {nelec}")
dim = comb(norb, nelec[0]) * comb(norb, nelec[1])
print(f"Hilbert space dimension: {dim}")
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
g2e = pyscf.ao2mo.full(mol, mf.mo_coeff) # compressed
g2e_full = pyscf.ao2mo.restore(1, g2e, norb) # not compressed; chemist's notation
# recall: chemist notation (pq|rs) = int d1 d2 \bar phi_p(1) phi_q(1) 1/r_12 \bar phi_r(2) phi_s(2) = <p, r| W |q, s>
ecore = mol.energy_nuc()

# --- Step 1.2: Run DMRG ---
solver = Block2DMRGSolver( # handles nelec as integer or pair
    h1e=h1e, g2e=g2e, ecore=ecore,
    n_elec=nelec, spin=mol.spin
)
# --- Step 1.2: get a random mps ---
# random mps
mps = solver.driver.get_random_mps(tag='RAND', bond_dim=5, nroots=1)

converged SCF energy = -74.9646625391309
Number of orbitals: 7
Number of elecs: (5, 5)
Hilbert space dimension: 441


In [6]:
# ============================================================================
# SECTION 2: Get 1, ..., 4-RDMs from the mps
# ============================================================================

# --- Step 2.1: Extract RDMs from MPS ---

# get rdms
rdm1_a, rdm1_b = solver.driver.get_1pdm(mps)
print("got 1rdm")
rdm2_aa, rdm2_ab, rdm2_bb = solver.driver.get_2pdm(mps)
print("got 2rdm")
rdm3_aaa, rdm3_aab, rdm3_abb, rdm3_bbb = solver.driver.get_3pdm(mps)
print("got 3rdm")
rdm4_aaaa, rdm4_aaab, rdm4_aabb, rdm4_abbb, rdm4_bbbb = solver.driver.get_4pdm(mps)
print("got 4rdm")

# --- Step 2.2: Spin-summed rdms ---

rdm1 = rdm1_a + rdm1_b
rdm2 = rdm2_aa + rdm2_bb + rdm2_ab + rdm2_ab.transpose(1, 0, 3, 2)


rdm3 = (
    rdm3_aaa 
    + rdm3_aab + rdm3_aab.transpose(0, 2, 1, 4, 3, 5) + rdm3_aab.transpose(2, 1, 0, 5, 4, 3) 
    + rdm3_abb + rdm3_abb.transpose(1, 0, 2, 3, 5, 4) + rdm3_abb.transpose(2, 1, 0, 5, 4, 3) 
    + rdm3_bbb
)

rdm4 = (
    # 0 betas (k=0)
    rdm4_aaaa
    
    # 1 beta (k=1)
    # Target beta positions: (3,), (2,), (1,), (0,)
    + rdm4_aaab
    + rdm4_aaab.transpose(0, 1, 3, 2,     5, 4, 6, 7) # (0, 1, 3, 2, 5, 4, 6, 7) 
    + rdm4_aaab.transpose(0, 3, 2, 1,     6, 5, 4, 7) # (0, 3, 1, 2, 5, 6, 4, 7) 
    + rdm4_aaab.transpose(3, 1, 2, 0,     7, 5, 6, 4) # (3, 0, 1, 2, 5, 6, 7, 4) 
    
    # 2 betas (k=2)
    # Target beta positions: (2,3), (1,3), (1,2), (0,3), (0,2), (0,1)
    + rdm4_aabb
    + rdm4_aabb.transpose(0, 2, 1, 3,     4, 6, 5, 7) # (0, 2, 1, 3, 4, 6, 5, 7) 
    + rdm4_aabb.transpose(0, 3, 2, 1,     6, 5, 4, 7) # (0, 2, 3, 1, 6, 4, 5, 7) 
    + rdm4_aabb.transpose(2, 1, 0, 3,     4, 7, 6, 5) # (2, 0, 1, 3, 4, 6, 7, 5) 
    + rdm4_aabb.transpose(3, 1, 2, 0,     7, 5, 6, 4) # (2, 0, 3, 1, 6, 4, 7, 5) 
    + rdm4_aabb.transpose(2, 3, 0, 1,     6, 7, 4, 5) # (2, 3, 0, 1, 6, 7, 4, 5)
    
    # 3 betas (k=3)
    # Target beta positions: (1,2,3), (0,2,3), (0,1,3), (0,1,2)
    + rdm4_abbb
    + rdm4_abbb.transpose(1, 0, 2, 3,     4, 5, 7, 6) # (1, 0, 2, 3, 4, 5, 7, 6) 
    + rdm4_abbb.transpose(2, 1, 0, 3,     4, 7, 6, 5) # (1, 2, 0, 3, 4, 7, 5, 6) 
    + rdm4_abbb.transpose(3, 1, 2, 0,     7, 5, 6, 4) # (1, 2, 3, 0, 7, 4, 5, 6) 
    
    # 4 betas (k=4)
    + rdm4_bbbb
)

got 1rdm
got 2rdm
got 3rdm
got 4rdm


In [ ]:
# --- Verify 3RDM ---
import numpy as np
import itertools
import ffsim

# fci state
fci_state = solver.to_ci_vector(ket=mps)

# fermionic operators (alpha and beta)
cre_a, des_a = ffsim.cre_a, ffsim.des_a
cre_b, des_b = ffsim.cre_b, ffsim.des_b

# --- Verify 3RDM ---
orbitals = [3, 6, 1, 2]
for i, j, k, l, m, n in itertools.product(orbitals, repeat=6):
    expected = 0.0
    # Sum over all spin combinations for the three creation operators
    for spin_i in [cre_a, cre_b]:
        for spin_j in [cre_a, cre_b]:
            for spin_k in [cre_a, cre_b]:
                # Determine the corresponding annihilation operators
                # For des(l): same spin as cre(i)
                des_l = des_a if spin_k == cre_a else des_b
                # For des(m): same spin as cre(j)
                des_m = des_a if spin_j == cre_a else des_b
                # For des(n): same spin as cre(k)
                des_n = des_a if spin_i == cre_a else des_b

                # Construct the operator: cre(i) cre(j) cre(k) des(l) des(m) des(n)
                op = ffsim.linear_operator(
                    ffsim.FermionOperator({
                        (spin_i(i), spin_j(j), spin_k(k), des_l(l), des_m(m), des_n(n)): 1
                    }),
                    norb, nelec
                )
                expected += np.vdot(fci_state, op @ fci_state).real  # Assuming real-valued expectation

    # Compare with your implemented rdm3
    if not np.isclose(rdm3[i, j, k, l, m, n], expected):
        print(f"3RDM Mismatch at ({i},{j},{k},{l},{m},{n}): "
              f"Expected {expected}, Got {rdm3[i, j, k, l, m, n]}")
#    else:
#        print(f"All good at ({i},{j},{k},{l},{m},{n})")


In [8]:
"""
# --- Verify 4RDM ---
orbitals = [1, 2, 3, 4]
for i, j, k, l, m, n, p, q in itertools.product(orbitals, repeat=8):
    expected = 0.0
    # Sum over all spin combinations for the four creation operators
    for spin_i in [cre_a, cre_b]:
        for spin_j in [cre_a, cre_b]:
            for spin_k in [cre_a, cre_b]:
                for spin_l in [cre_a, cre_b]:
                    # Determine the corresponding annihilation operators
                    des_m = des_a if spin_l == cre_a else des_b
                    des_n = des_a if spin_k == cre_a else des_b
                    des_p = des_a if spin_j == cre_a else des_b
                    des_q = des_a if spin_i == cre_a else des_b

                    # Construct the operator: cre(i) cre(j) cre(k) cre(l) des(q) des(p) des(n) des(m)
                    op = ffsim.linear_operator(
                        ffsim.FermionOperator({
                            (spin_i(i), spin_j(j), spin_k(k), spin_l(l),
                             des_m(m), des_n(n), des_p(p), des_q(q)): 1
                        }),
                        norb, nelec
                    )
                    expected += np.vdot(fci_state, op @ fci_state).real

    # Compare with your implemented rdm4
    if not np.isclose(rdm4[i, j, k, l, m, n, p, q], expected):
        print(f"4RDM Mismatch at ({i},{j},{k},{l},{m},{n},{p},{q}): "
              f"Expected {expected}, Got {rdm4[i, j, k, l, m, n, p, q]}")
#    else:
#        print(f"All good at ({i},{j},{k},{l},{m},{n},{p},{q})")
"""

'\n# --- Verify 4RDM ---\norbitals = [1, 2, 3, 4]\nfor i, j, k, l, m, n, p, q in itertools.product(orbitals, repeat=8):\n    expected = 0.0\n    # Sum over all spin combinations for the four creation operators\n    for spin_i in [cre_a, cre_b]:\n        for spin_j in [cre_a, cre_b]:\n            for spin_k in [cre_a, cre_b]:\n                for spin_l in [cre_a, cre_b]:\n                    # Determine the corresponding annihilation operators\n                    des_m = des_a if spin_l == cre_a else des_b\n                    des_n = des_a if spin_k == cre_a else des_b\n                    des_p = des_a if spin_j == cre_a else des_b\n                    des_q = des_a if spin_i == cre_a else des_b\n\n                    # Construct the operator: cre(i) cre(j) cre(k) cre(l) des(q) des(p) des(n) des(m)\n                    op = ffsim.linear_operator(\n                        ffsim.FermionOperator({\n                            (spin_i(i), spin_j(j), spin_k(k), spin_l(l),\n              

In [9]:
# ============================================================================
# SECTION 3: Squared commutator norm 
# ============================================================================

# checked numerically, it's almost surely correct

$$
\text{squared commutator exp value}(h,g_{chem},D^{(1)},D^{(2)},D^{(3)},D^{(4)},C) = A + B + B^* + C
$$

with $g_{pqrs} = g_{chem, prqs}$ and

$$
A=
\sum_{t,t'\in C}\sum_{p,q}\sum_{p',q'}
h_{pq}\,h_{p'q'}\,
(-\delta_{pt}+\delta_{qt})\,(-\delta_{p't'}+\delta_{q't'})\,
\Bigl(
D^{(2)}_{p\,p'\,q'\,q}
+\delta_{q p'}\,D^{(1)}_{p\,q'}
\Bigr)
$$

$$
B = \frac12
\sum_{t,t'\in C}\sum_{p,q}\sum_{p',q',r',s'}
h_{pq}\,g_{p'q'r's'}\,
(-\delta_{pt}+\delta_{qt})\,
(-\delta_{p't'}-\delta_{q't'}+\delta_{s't'}+\delta_{r't'})\,
\Bigl[
D^{(3)}_{p\,p'\,q'\,s'\,r'\,q}
+\delta_{q p'}\,D^{(2)}_{p\,q'\,s'\,r'}
+\delta_{q q'}\,D^{(2)}_{p\,p'\,r'\,s'}
\Bigr]
$$

$$
C =
\frac14
\sum_{t,t'\in C}\sum_{p,q,r,s}\sum_{p',q',r',s'}
g_{pqrs}\,g_{p'q'r's'}\,
(-\delta_{pt}-\delta_{qt}+\delta_{st}+\delta_{rt})\,
(-\delta_{p't'}-\delta_{q't'}+\delta_{s't'}+\delta_{r't'})\,
\Bigl[
D^{(4)}_{p\,q\,p'\,q'\,s'\,r'\,s\,r}


+\delta_{s p'}\,D^{(3)}_{p\,q\,q'\,s'\,r'\,r}
+\delta_{s q'}\,D^{(3)}_{p\,q\,p'\,r'\,s'\,r}
+\delta_{r p'}\,D^{(3)}_{p\,q\,q'\,s'\,s\,r'}
+\delta_{r q'}\,D^{(3)}_{p\,q\,p'\,r'\,s\,s'}

+\delta_{r p'}\delta_{s q'}\,D^{(2)}_{p\,q\,s'\,r'}
+\delta_{r q'}\delta_{s p'}\,D^{(2)}_{p\,q\,r'\,s'}
\Bigr]
$$

In [10]:
import numpy as np

# the three implementations in this cell agree!

# function that returns <[H, N_C]^2> given 1elec orbitals h, 2elec orbitals g (chemist's notation), 1-, 2-, 3-, 4-rdms (spin-summed), cluster of orbitals C as a list.
import numpy as np


def squared_commutator_exp_value(h, g_chem, D1, D2, D3, D4, C, return_terms=False):
    """
    Implements

        F = A + B + B* + C

    with the tensor contractions exactly as in the corrected LaTeX.

    Index conventions used literally as written in the formula:
      A:
        D2[p, pp, qp, q]
        D1[p, qp]

      B:
        D3[p, pp, qp, sp, rp, q]
        D2[p, qp, sp, rp]
        D2[p, pp, rp, sp]

      C (corrected version -- note the r<->s, r'<->s' swap on every D and
      every delta relative to the original/mistaken formula; the g tensors
      are NOT swapped):
        D4[p, q, pp, qp, sp, rp, s, r]
        D3[p, q, qp, sp, rp, r]
        D3[p, q, pp, rp, sp, r]
        D3[p, q, qp, sp, s, rp]
        D3[p, q, pp, rp, s, sp]
        D2[p, q, sp, rp]
        D2[p, q, rp, sp]

    Key simplification: since the (-delta+delta)-type factors depend on
    (p, q, t) resp. (p', q', r', s', t') separately from everything else,
    the sums over t, t' in C factor and can be pre-summed into simple
    "indicator" combinations:

        S1[p, q]       = sum_{t in C} (-delta(p,t) + delta(q,t))
                        = -I[p] + I[q]

        S2[p, q, r, s] = sum_{t in C} (-delta(p,t) - delta(q,t)
                                        + delta(s,t) + delta(r,t))
                        = -I[p] - I[q] + I[s] + I[r]

    where I is the indicator vector of the index set C. This turns the
    (t, t')-nested sums into ordinary tensor contractions (einsum).
    """
    h = np.asarray(h)
    g = np.asarray(g_chem).transpose(0, 2, 1, 3)
    D1 = np.asarray(D1)
    D2 = np.asarray(D2)
    D3 = np.asarray(D3)
    D4 = np.asarray(D4)

    norb = h.shape[0]
    C = tuple(C)

    dtype = np.result_type(h, g, D1, D2, D3, D4, np.complex128)

    # Indicator vector for the index set C
    idx = np.array(C, dtype=int)
    I = np.zeros(norb, dtype=dtype)
    I[idx] = 1.0

    # -----------------------
    # Aggregated delta-sums
    # -----------------------
    # S1[p, q] = -I[p] + I[q]
    S1 = -I[:, None] + I[None, :]

    # S2[p, q, r, s] = -I[p] - I[q] + I[s] + I[r]
    S2 = (-I[:, None, None, None] - I[None, :, None, None]
          + I[None, None, None, :] + I[None, None, :, None])

    # M[p, q] = h[p, q] * S1[p, q]
    M = h * S1

    # Gt[p, q, r, s] = g[p, q, r, s] * S2[p, q, r, s]
    # (same tensor is reused for both primed and unprimed index groups)
    Gt = g * S2

    # -----------------------
    # A term
    # -----------------------
    # A = sum_{p,q,p',q'} h_pq h_p'q' S1_pq S1_p'q' (D2[p,p',q',q] + delta(q,p') D1[p,q'])
    A1 = np.einsum('pq,rs,prsq->', M, M, D2, optimize=True)
    A2 = np.einsum('pq,qr,pr->', M, M, D1, optimize=True)
    A = A1 + A2

    # -----------------------
    # B term
    # -----------------------
    # B = 1/2 sum h_pq g_p'q'r's' S1_pq S2_p'q'r's' [
    #        D3[p,p',q',s',r',q] + delta(q,p') D2[p,q',s',r'] + delta(q,q') D2[p,p',r',s'] ]
    B1 = np.einsum('pq,ijkl,pijlkq->', M, Gt, D3, optimize=True)
    B2 = np.einsum('pq,qjkl,pjlk->', M, Gt, D2, optimize=True)
    B3 = np.einsum('pq,iqkl,pikl->', M, Gt, D2, optimize=True)
    B = 0.5 * (B1 + B2 + B3)

    # -----------------------
    # C term (corrected)
    # -----------------------
    # C = 1/4 sum g_pqrs g_p'q'r's' S2_pqrs S2_p'q'r's' [
    #        D4[p,q,p',q',s',r',s,r]
    #      + delta(s,p') D3[p,q,q',s',r',r]
    #      + delta(s,q') D3[p,q,p',r',s',r]
    #      + delta(r,p') D3[p,q,q',s',s,r']
    #      + delta(r,q') D3[p,q,p',r',s,s']
    #      + delta(r,p') delta(s,q') D2[p,q,s',r']
    #      + delta(r,q') delta(s,p') D2[p,q,r',s'] ]

    C1 = np.einsum('pqrs,ijkl,pqijlksr->', Gt, Gt, D4, optimize=True)
    C2 = np.einsum('pqrs,sjkl,pqjlkr->', Gt, Gt, D3, optimize=True)
    C3 = np.einsum('pqrs,iskl,pqiklr->', Gt, Gt, D3, optimize=True)
    C4 = np.einsum('pqrs,rjkl,pqjlsk->', Gt, Gt, D3, optimize=True)
    C5 = np.einsum('pqrs,irkl,pqiksl->', Gt, Gt, D3, optimize=True)
    C6 = np.einsum('pqrs,rskl,pqlk->', Gt, Gt, D2, optimize=True)
    C7 = np.einsum('pqrs,srkl,pqkl->', Gt, Gt, D2, optimize=True)
    CT = 0.25 * (C1 + C2 + C3 + C4 + C5 + C6 + C7)

    F = A + B + np.conj(B) + CT

    if return_terms:
        return A, B, CT
    else:
        return F

In [11]:
# same thing as above, but manual

def squared_commutator_exp_value_expected(h1_linop, h2_linop, fci_state, cluster, return_terms=False):


    # Construct the spin-summed number operator for the specified orbitals:
    # N = sum_{i in orbitals} (a^\dagger_{i,α} a_{i,α} + a^\dagger_{i,β} a_{i,β})
    number_op_terms = {}
    for i in cluster:
        number_op_terms[(cre_a(i), des_a(i))] = 1.0
        number_op_terms[(cre_b(i), des_b(i))] = 1.0

    num_operator = ffsim.FermionOperator(number_op_terms)
    num_linop = ffsim.linear_operator(num_operator, norb=norb, nelec=nelec)

    commutator1 = h1_linop @  num_linop - num_linop @ h1_linop
    commutator2 = h2_linop @  num_linop - num_linop @ h2_linop

    A = np.vdot(fci_state, commutator1 @ commutator1 @ fci_state)
    B = np.vdot(fci_state, commutator1 @ commutator2 @ fci_state)
    C = np.vdot(fci_state, commutator2 @ commutator2 @ fci_state)

    F = A + B + np.conjugate(B) + C
    if return_terms:
        return A, B, C
    else:
        return F


In [12]:
# ============================================================================
# SECTION 4: Test squared commutator norm functions against exact one.
# ============================================================================

clusters = [[0, 1, 2], [4, 6, 2], [1, 6, 2], [3], [5, 3], [0, 2, 4, 6], range(1), range(2), range(3), range(4), range(5), range(6), range(7)]

h1e_zeros = np.zeros((norb, norb))
g2e_full_zeros = np.zeros((norb, norb, norb, norb))


# Construct the ffsim 1P Hamiltonian using chemist notation integrals
hamiltonian1 = ffsim.MolecularHamiltonian(
    one_body_tensor=h1e, 
    two_body_tensor=g2e_full_zeros, # here: chemist notation
    constant=ecore
)

# Interaction
hamiltonian2 = ffsim.MolecularHamiltonian(
    one_body_tensor=h1e_zeros, 
    two_body_tensor=g2e_full, # here: chemist notation
    constant=ecore
)

# Generate the linear operators acting on the (N_alpha, N_beta) subspace
h1_linop = ffsim.linear_operator(hamiltonian1, norb, nelec)
h2_linop = ffsim.linear_operator(hamiltonian2, norb, nelec)

for cluster in clusters:
    A_rdm, B_rdm, C_rdm = squared_commutator_exp_value(h1e, g2e_full, rdm1, rdm2, rdm3, rdm4, cluster, return_terms=True)
    A_op, B_op, C_op = squared_commutator_exp_value_expected(h1_linop, h2_linop, fci_state, cluster, return_terms=True)

    assert (A_rdm - A_op).round(10) == 0, f"A diff: {(A_rdm - A_op).round(10)}"
    assert (B_rdm - B_op).round(10) == 0, f"B diff: {(B_rdm - B_op).round(10)}"
    assert (C_rdm - C_op).round(10) == 0, f"C diff: {(C_rdm - C_op).round(10)}"

In [13]:
# ============================================================================
# SECTION 5: Gemini's implementation - BUG
# ============================================================================
import numpy as np
import jax
import jax.numpy as jnp
from typing import Callable
from src.cluster_number_operators import params_to_U_jax

def number_variance_cost(h1e, g2e_full, D, Gamma, rdm3, rdm4, cluster_matrix, with_ghost=False) -> Callable:
    """Builds cost function returning a commutator-based score of an orbital rotation. 
    Because the commutator of a 1- and 2-body Hamiltonian with n_p is also a 1- and 2-body 
    operator, only up to the 4th rdms are needed.

    Args:
        h1e: 1-electron integrals in MO basis
        g2e_full: 2-electron integrals in MO basis; chemist's notation   
        D (ndarray): The spin-summed 1-reduced density matrix (1-RDM) of an underlying state psi.
        Gamma (ndarray): The spin-summed 2-reduced density matrix (2-RDM) of psi.
        rdm3 (ndarray): The spin-summed 3-reduced density matrix (3-RDM) of psi.
        rdm4 (ndarray): The spin-summed 4-reduced density matrix (4-RDM) of psi.
        cluster_matrix (ndarray): A binary matrix/list defining the orbital clusters.
        with_ghost (bool): set to True to add a cluster with all orbitals that are not in cluster_matrix. Defaults to False.

    Returns:
        Callable: A function `f(x)` that returns the quantity -\sum_C <psi(U)|[H(U), N_C]^2|psi(U)>.
    """
    norb = h1e.shape[0]

    # Use NumPy to precompute the huge M tensor ONCE. 
    # This prevents JAX from unrolling/tracing an O(N^8) graph.
    h = np.array(h1e)
    g = np.array(g2e_full)
    D_np = np.array(D)
    Gamma_np = np.array(Gamma)
    rdm3_np = np.array(rdm3)
    rdm4_np = np.array(rdm4)
    delta = np.eye(norb)

    # 1. Base commutator coefficients: A_ab = [H, E^a_b] = u_tensor + V_tensor
    # u_tensor[a, b, i, j] represents the 1-body part
    u_tensor = np.einsum('bj,ia->abij', delta, h) - np.einsum('ai,bj->abij', delta, h)
    
    # V_tensor[a, b, i, j, k, l] represents the 2-body part.
    # Matches chemist notation g[p,q,r,s] -> operator E^{pr}_{qs} (p=i, r=j, q=k, s=l)
    V_tensor = 0.5 * (
        np.einsum('bk,iajl->abijkl', delta, g) +
        np.einsum('bl,ikja->abijkl', delta, g) -
        np.einsum('ai,bkjl->abijkl', delta, g) -
        np.einsum('aj,ikbl->abijkl', delta, g)
    )

    # 2. Build M_{abcd} = < A_ab A_cd > exactly using Wick's expansion mapped to 1..4 RDMs
    # M11: < V1 V1 >
    M11 = (np.einsum('abij,cdjl,il->abcd', u_tensor, u_tensor, D_np, optimize=True) +
           np.einsum('abij,cdkl,iklj->abcd', u_tensor, u_tensor, Gamma_np, optimize=True))
           
    # M12: < V1 V2 >
    M12 = (np.einsum('abij,cdjlmn,ilnm->abcd', u_tensor, V_tensor, Gamma_np, optimize=True) -
           np.einsum('abij,cdklmn,iknm->abcd', u_tensor, V_tensor, Gamma_np, optimize=True) +
           np.einsum('abij,cdklmn,ikljnm->abcd', u_tensor, V_tensor, rdm3_np, optimize=True))
           
    # M21: < V2 V1 >
    M21 = (np.einsum('abijkl,cdkn,ijln->abcd', V_tensor, u_tensor, Gamma_np, optimize=True) -
           np.einsum('abijkl,cdln,ijkn->abcd', V_tensor, u_tensor, Gamma_np, optimize=True) +
           np.einsum('abijkl,cdmn,ijmlkn->abcd', V_tensor, u_tensor, rdm3_np, optimize=True))
           
    # M22: < V2 V2 >
    M22 = (np.einsum('abijkl,cdklwz,ijwz->abcd', V_tensor, V_tensor, Gamma_np, optimize=True) -
           np.einsum('abijkl,cdlkwz,ijwz->abcd', V_tensor, V_tensor, Gamma_np, optimize=True) -
           np.einsum('abijkl,cdkywz,ijylwz->abcd', V_tensor, V_tensor, rdm3_np, optimize=True) +
           np.einsum('abijkl,cdxkwz,ijxlwz->abcd', V_tensor, V_tensor, rdm3_np, optimize=True) +
           np.einsum('abijkl,cdlywz,ijykwz->abcd', V_tensor, V_tensor, rdm3_np, optimize=True) -
           np.einsum('abijkl,cdxlwz,ijxkwz->abcd', V_tensor, V_tensor, rdm3_np, optimize=True) +
           np.einsum('abijkl,cdxywz,ijxylkwz->abcd', V_tensor, V_tensor, rdm4_np, optimize=True))

    M_np = M11 + M12 + M21 + M22
    
    # 3. Transfer the precomputed O(N^4) footprint tensor to JAX
    M_jax = jnp.array(M_np)
    
    # 4. Resolve the static cluster projectors
    cluster_indices = get_cluster_indices(cluster_matrix, norb, with_ghost=with_ghost)
    P_C_masks = []
    for cluster in cluster_indices:
        mask = np.zeros(norb)
        mask[cluster] = 1.0
        P_C_masks.append(jnp.array(mask))

    def f(x: jnp.ndarray) -> float:
        # Create orbital unitary
        U = params_to_U_jax(x, norb)
        Uc = U.conj()
        
        total_score = 0.0
        for P_C in P_C_masks:
            # Construct the transformed cluster number matrix B = U^dagger P_C U
            # (efficient diagonal multiplication via einsum)
            B = jnp.einsum('pi,p,pj->ij', Uc, P_C, U)
            
            # Evaluate the <V_C^2> expectation using the precomputed M
            expected_V_squared = jnp.einsum('ab,cd,abcd->', B, B, M_jax)
            
            total_score += expected_V_squared.real
            
        # The commutator expectation represents <V^2>.
        # Because V is strictly anti-Hermitian, V^2 is negative-semidefinite, making <V^2> <= 0.
        # Returning -total_score ensures the yielded quantity is non-negative.
        return -total_score
        
    return f

In [14]:
# ============================================================================
# SECTION 4: Build old and new cost functions
# ============================================================================

from src.cluster_number_operators import number_matrix_to_operators, get_cluster_indices
import optimize_symmetries
from optimize_symmetries import commutator_cost
optimize_symmetries.pyscf = pyscf # in case try-except import code in optimize_symmetries fails
optimize_symmetries.ffsim = ffsim # in case try-except import code in optimize_symmetries fails

number_operators = number_matrix_to_operators(cluster_matrix, norb, nelec) # no ghost

moldata = pyscf.lib.chkfile.load_mol(mf.chkfile)
mf_update = pyscf.scf.RHF(mol) # why again?
mf_update.update_from_chk(mf.chkfile)
moldata_ffsim = ffsim.MolecularData.from_scf(mf_update)
g2e_full = pyscf.ao2mo.restore(1, g2e, norb) # not compressed; chemist's notation

f_old = commutator_cost(moldata_ffsim, number_operators, fci_state)
f_new = number_variance_cost(h1e, g2e_full, rdm1, rdm2, rdm3, rdm4, cluster_matrix, with_ghost=False)

In [15]:
# ============================================================================
# SECTION 5: Compare
# ============================================================================

from math import comb

number_tests = 10
length_x = comb(norb, 2)
xs = [np.random.rand(length_x) for _ in range(number_tests)]
for x in xs:
    print(f_old(x))
    print(f_new(x))
    print()

103.1722804838719
155.87954464926983

109.55853807440644
156.47133373650234

216.14173524629072
311.10004801315273

229.67054216860606
365.40256087019634

230.56011796730047
378.8014258623284

298.969426628074
435.9597434187485

205.37375924296737
290.8415493264738

206.01312762878592
288.972915771222

287.1453655625267
449.83340783526205

107.09018874789173
144.5767152129249

